In [17]:
from enum import Enum
from datetime import datetime
from datetime import date
 
class QuotationType(Enum):
    NOMINAL = 1
    AMOUNT = 2
 
class TransactionType(Enum):
    CASH = 1
    SECURITY = 2
 
class InterestType(Enum):
    ACT_ACT = 1
    _30_360 = 2
 
class PaymentFrequency(Enum):
    MONTH = 1
    YEAR = 2
    END_DATE = 3


In [18]:
class FinancialInstrument:
    def __init__(self, instrument_id, instrument_name, start_date, end_date, currency, trade_minimum, lot_size, quotation_type):
        self.instrument_id = instrument_id
        self.instrument_name = instrument_name
        self.start_date = start_date
        self.end_date = end_date
        self.currency = currency
        self.trade_minimum = trade_minimum
        self.lot_size = lot_size
        self.quotation_type = quotation_type
 
    def calculate_value(self):
        pass  # calculate the value of the financial instrument
 


In [19]:
class Bond(FinancialInstrument):
    def __init__(self, instrument_id, instrument_name, start_date, end_date, currency, trade_minimum, lot_size, quotation_type, interest_percentage, interest_type=InterestType.ACT_ACT, payment_frequency=PaymentFrequency.YEAR, maturity_date=None, issue_price=100):
        super().__init__(instrument_id, instrument_name, start_date, end_date, currency, trade_minimum, lot_size, quotation_type)
        self.interest_percentage = interest_percentage
        self.interest_type = interest_type
        self.payment_frequency = payment_frequency
        self.maturity_date = maturity_date
        self.issue_price = issue_price
 
    def calculate_interest(self):
        pass  # calculate the interest for the bond
 


In [20]:
class TradingEnvironment:
    def __init__(self):
        self.instruments = {}  # Opslag voor alle instrumenten

    def add_instrument(self, instrument):
        self.instruments[instrument.instrument_id] = instrument

    def check_instrument_exists(self, instrument_id):
        return instrument_id in self.instruments

    def create_bond(self, *args, **kwargs):
        bond = Bond(*args, **kwargs)
        self.add_instrument(bond)
        return bond

    def add_securities_transaction(self, portfolio, instrument_id, amount_nominal, transaction_type, price):
        if self.check_instrument_exists(instrument_id):
            new_transaction = SecuritiesTransaction(instrument_id, amount_nominal, transaction_type, price)
            portfolio.securities_transactions.append(new_transaction)
        else:
            print(f"Instrument {instrument_id} bestaat niet.")

In [21]:
class Transaction:
    def __init__(self, transaction_type, transaction_date, value_date=None, transaction_status='approved', dummy=False):
        self.transaction_type = transaction_type
        self.transaction_date = transaction_date
        self.value_date = value_date
        self.transaction_status = transaction_status
        self.dummy = dummy
 
    def process_transaction(self):
        pass  # process the transaction
 


In [22]:
class CashTransaction(Transaction):
    def __init__(self, amount, transaction_type, currency='EUR', transaction_date=datetime.today(), value_date=None, transaction_status='approved', dummy=False):
        super().__init__(transaction_type, transaction_date, value_date, transaction_status, dummy)
        self.currency = currency
        self.amount = amount
 
    def calculate_amount(self):
        pass  # calculate the amount for the cash transaction
 


In [23]:
class SecuritiesTransaction(Transaction):
    def __init__(self, instrument_id, amount_nominal, transaction_type, price, transaction_date=datetime.today(), value_date=None, transaction_status='approved', dummy=False):
        super().__init__(transaction_type, transaction_date, value_date, transaction_status, dummy)
        self.instrument_id = instrument_id
        self.amount_nominal = amount_nominal
        self.price = price
 
    def calculate_transaction_value(self):
        return self.amount_nominal * self.price  # voorlopige berekening
 
    def calculate_nominal_amount(self):
        pass  # calculate the nominal amount for the securities transaction
 


In [24]:
class Portfolio:
    def __init__(self, portfolio_id, owner, start_date=datetime.today()):
        self.portfolio_id = portfolio_id
        self.start_date = start_date
        self.end_date = None
        self.owner = owner
        self.cash_accounts = []
        self.securities_transactions = []
        self.instruments = {}  # dictionary om FinancialInstrument-objecten op te slaan
 
    def add_cash_account(self, currency):
        if any(account.currency == currency for account in self.cash_accounts):
            print("A cash account with this currency already exists.")
            return
        account_id = self.portfolio_id + currency
        new_account = CashAccount(account_id, currency)
        self.cash_accounts.append(new_account)
 
    def list_cash_accounts(self):
        for account in self.cash_accounts:
            print(account.account_id)
 
    def list_all_transactions(self):
        for account in self.cash_accounts:
            for transaction in account.transactions:
                print(vars(transaction))
        for transaction in self.securities_transactions:
            print(vars(transaction))
 
    def list_transactions_for_instrument(self, instrument_id):
        for transaction in self.securities_transactions:
            if transaction.instrument_id == instrument_id:
                print(vars(transaction))
 
    def add_cash_transaction(self, amount, transaction_type, currency='EUR'):
        new_transaction = CashTransaction(amount, transaction_type, currency)
        # Hier voegen we de transactie toe aan de juiste CashAccount
        for account in self.cash_accounts:
            if account.currency == currency:
                account.transactions.append(new_transaction)
                return
        print("Geen account gevonden met de gegeven valuta.")
 
    def add_securities_transaction(self, instrument_id, amount_nominal, transaction_type, price):
        # Controleer of het instrument bestaat en of de einddatum nog niet is verstreken
        instrument = self.instruments.get(instrument_id)
        if not instrument:
            print(f"Instrument {instrument_id} bestaat niet.")
            return
        if datetime.today() > instrument.end_date:
            print(f"Het instrument {instrument_id} is verlopen.")
            return
 
        new_transaction = SecuritiesTransaction(instrument_id, amount_nominal, transaction_type, price)
        self.securities_transactions.append(new_transaction)
 
    def add_instrument(self, instrument):
        self.instruments[instrument.instrument_id] = instrument
 
    def create_bond(self, instrument_id, instrument_name, start_date, end_date, currency, trade_minimum, lot_size, quotation_type, interest_percentage, interest_type=InterestType.ACT_ACT, payment_frequency=PaymentFrequency.YEAR, maturity_date=None, issue_price=100):
        bond = Bond(instrument_id, instrument_name, start_date, end_date, currency, trade_minimum, lot_size, quotation_type, interest_percentage, interest_type, payment_frequency, maturity_date, issue_price)
        self.add_instrument(bond)
        return bond
 
    def check_instrument(self, instrument_id):
        instrument = self.instruments.get(instrument_id)
        if not instrument:
            print(f"Instrument {instrument_id} bestaat niet.")
            return
        if date.today() > instrument.end_date:
            print(f"Het instrument {instrument_id} is verlopen.")
        else:
            print(f"Het instrument {instrument_id} is geldig en niet verlopen.")


In [25]:
 
class CashAccount:
    def __init__(self, account_id, currency='EUR', interest_percentage=0):
        self.account_id = account_id
        self.currency = currency
        self.interest_percentage = interest_percentage
        self.transactions = []
       
    def calculate_balance(self):
        pass  # calculate the balance for the cash account


In [26]:
class Prices:
    def __init__(self, instrument_id, price_date, price, currency='EUR'):
        self.instrument_id = instrument_id
        self.price_date = price_date
        self.price = price
        self.currency = currency
 
    def get_latest_price(self):
        pass  # get the latest price for the financial instrument


In [27]:
 
portfolio = Portfolio("001", "John Doe")
# Voeg een cashrekening toe
portfolio.add_cash_account("USD")
portfolio.add_cash_account("EUR")
 
# Lijst alle cashrekeningen op
portfolio.list_cash_accounts()  # Dit zal afdrukken: 001USD


trading_env = TradingEnvironment()
trading_env.create_bond('123', 'Bond Name', '2022-01-01', '2030-12-31', 'EUR', 1000, 1000, 1, QuotationType.NOMINAL, 5, issue_price=101)

trading_env.add_securities_transaction(portfolio, '123', 500, 'koop', 1.5)
#portfolio.add_cash_transaction(buy_bond.calculate_transaction_value(), 'af', 'USD')
 
# Voeg een cashtransactie toe
portfolio.add_cash_transaction(1000, 'bij', 'USD')
 
#buy_bond = SecuritiesTransaction('123', 500, 'koop', 1.5)
#portfolio.add_securities_transaction('123', 500, 'koop', 1.5)
 
 
#portfolio.list_all_transactions()
 

# Lijst alle transacties voor instrument '123'
portfolio.list_transactions_for_instrument('123')

001USD
001EUR
{'transaction_type': 'koop', 'transaction_date': datetime.datetime(2024, 4, 27, 11, 42, 1, 305225), 'value_date': None, 'transaction_status': 'approved', 'dummy': False, 'instrument_id': '123', 'amount_nominal': 500, 'price': 1.5}
